# マージ済みモデルの作成と vLLM DLC エンドポイントでの配信（MTP なし）

`install_guide/07_inference_and_merge.md` ステップ2 の Notebook です。

1. 学習ジョブの LoRA アダプタをベースモデルにマージした HF 形式のモデルを、SageMaker Training Job（`src/inference/merge_adapter.py`）で作る
2. AWS の vLLM Deep Learning Container（`vllm:server-sagemaker-cuda-v2.5`、vLLM 0.30.0）で SageMaker リアルタイムエンドポイントにデプロイする
3. Chat Completions 形式で呼び出し、ステップ1 のアダプタ付き生成と比較する
4. エンドポイントを削除する

エンドポイントは起動中ずっと課金されるため、最後の削除セルを必ず実行してください。

**vLLM のインストールは不要です。** この Notebook（Studio の CPU カーネル）が使うのは `sagemaker` SDK と `boto3` だけで、モデルの実行は行いません。
vLLM は AWS が配布する vLLM DLC（`763104351884.dkr.ecr.<region>.amazonaws.com/vllm:server-sagemaker-cuda-v2.5`、vLLM 0.30.0 同梱）の
コンテナとしてエンドポイントのインスタンス上で動き、Notebook からは HTTP で呼び出すだけです。学習イメージにも Studio にも vLLM は入っていませんが、それで問題ありません。


In [ ]:
import os, json, time, tarfile, boto3, sagemaker
from sagemaker.pytorch import PyTorch
from sagemaker.model import Model
from sagemaker.predictor import Predictor
from sagemaker.serializers import JSONSerializer
from sagemaker.deserializers import JSONDeserializer

sess    = sagemaker.Session()
try:
    role = os.environ.get('SAGEMAKER_ROLE') or sagemaker.get_execution_role()
except Exception:
    raise SystemExit('ローカル実行時は SAGEMAKER_ROLE 環境変数に SageMaker 実行ロールの ARN を設定してください')
region  = boto3.Session().region_name
account = boto3.client('sts').get_caller_identity()['Account']
bucket  = sess.default_bucket()

train_image_uri = f'{account}.dkr.ecr.{region}.amazonaws.com/nemo-automodel-sagemaker:0.6.0-pt2.10-py313-cu130'
# AWS 公式の vLLM DLC (AL2023, vLLM 0.30.0)。763104351884 は DLC の公開アカウント
VLLM_IMAGE_TAG = 'server-sagemaker-cuda-v2.5'
vllm_image_uri = f'763104351884.dkr.ecr.{region}.amazonaws.com/vllm:{VLLM_IMAGE_TAG}'
print('sagemaker', sagemaker.__version__, '| region', region, '| bucket', bucket)
print('train image:', train_image_uri)
print('vllm image :', vllm_image_uri)


## 1. マージ対象のアダプタ

In [ ]:
training_job_name = ''   # 例: 'automodel-qwen35-cooking-lora-2026-09-24-05-06-27-958'
adapter_s3 = ''          # training_job_name を空にして直接指定してもよい

if training_job_name:
    desc = sess.sagemaker_client.describe_training_job(TrainingJobName=training_job_name)
    adapter_s3 = desc['ModelArtifacts']['S3ModelArtifacts']
    model_id   = json.loads(desc['HyperParameters'].get('model_id', '"Qwen/Qwen3.5-0.8B"'))
else:
    model_id = 'Qwen/Qwen3.5-0.8B'
if not adapter_s3:
    print('アダプタ未指定。セル 2 で merged_s3 を直接指定する場合はこのままで問題ありません')
val_s3 = f's3://{bucket}/automodel/cooking_basics/validation/'
print('adapter :', adapter_s3); print('model_id:', model_id)


## 2. マージジョブ

学習と同じイメージで `merge_adapter.py` を実行します。`peft` は `src/inference/requirements.txt` から入ります。
出力の `model.tar.gz` が、config.json を最上位に持つ HF 形式のフルモデルになります（`mtp.*` は含まれません）。

マージ済みの `model.tar.gz` が既にある場合は、次のセルの `merged_s3` にその S3 URI を書くとジョブをスキップします。


In [ ]:
merged_s3 = ''   # 既にマージ済みの model.tar.gz があれば S3 URI をここに書く (マージジョブをスキップ)
# 例: merged_s3 = f's3://{bucket}/automodel-merge-adapter-2026-09-25-00-30-25-987/output/model.tar.gz'

if not merged_s3:
    assert adapter_s3, 'セル 1 で training_job_name か adapter_s3 を指定してください'
    merge_estimator = PyTorch(
        image_uri=train_image_uri,
        entry_point='merge_adapter.py',
        source_dir='../src/inference',
        role=role,
        base_job_name='automodel-merge-adapter',
        instance_type='ml.g5.2xlarge',
        instance_count=1,
        volume_size=50,
        max_run=3600,
        hyperparameters={'model_id': model_id, 'dtype': 'bfloat16', 'verify': 1, 'num_samples': 3, 'max_new_tokens': 96},
        environment={'HF_HOME': '/tmp/hf', 'HF_TOKEN': os.environ.get('HF_TOKEN', ''), 'WANDB_MODE': 'disabled'},
    )
    merge_estimator.fit({'adapter': adapter_s3, 'validation': val_s3}, wait=True, logs='All')
    merge_job = merge_estimator.latest_training_job.name
    merged_s3 = sess.sagemaker_client.describe_training_job(TrainingJobName=merge_job)['ModelArtifacts']['S3ModelArtifacts']
    print('merge job :', merge_job)
print('merged    :', merged_s3)


## 3. vLLM DLC エンドポイントのデプロイ

`model_data` にマージ済みの `model.tar.gz` を渡すと `/opt/ml/model` に展開され、エントリポイントが自動で `--model /opt/ml/model` を付けます。
vLLM のフラグは `SM_VLLM_*` 環境変数で渡します。CUDA 13 系のため AL2 の GPU 推論 AMI（`al2-ami-sagemaker-inference-gpu-3-1`）を指定します。

`ml.g5.2xlarge` は在庫不足（`InsufficientInstanceCapacity`）になることがあるため、候補を順に試します。失敗したエンドポイントとその設定は自動で削除します。
候補はいずれも 24 GB 以上の GPU 1 枚で、0.8B モデルには十分です。使うには各インスタンスの `for endpoint usage` クォータが必要です。


In [ ]:
SERVED_NAME = 'qwen35-cooking-lora'
CANDIDATE_INSTANCES = ['ml.g5.2xlarge', 'ml.g6.2xlarge', 'ml.g5.xlarge', 'ml.g6e.2xlarge']   # 在庫が無ければ次を試す
sm = boto3.client('sagemaker')

vllm_env = {
    'SM_VLLM_SERVED_MODEL_NAME': SERVED_NAME,
    'SM_VLLM_MAX_MODEL_LEN': '4096',
    'SM_VLLM_DTYPE': 'bfloat16',
    'SM_VLLM_GPU_MEMORY_UTILIZATION': '0.85',
}

predictor = None
for inst in CANDIDATE_INSTANCES:
    endpoint_name = f'automodel-vllm-{int(time.time())}'
    vllm_model = Model(image_uri=vllm_image_uri, model_data=merged_s3, role=role, predictor_cls=Predictor, env=vllm_env)
    t0 = time.time()
    try:
        predictor = vllm_model.deploy(
            instance_type=inst,
            initial_instance_count=1,
            endpoint_name=endpoint_name,
            inference_ami_version='al2-ami-sagemaker-inference-gpu-3-1',
            container_startup_health_check_timeout=900,
            serializer=JSONSerializer(),
            deserializer=JSONDeserializer(),
        )
        instance_type_used = inst
        print(f'endpoint: {endpoint_name} | {inst} | InService まで {time.time()-t0:.0f}s')
        break
    except Exception as e:
        msg = str(e)
        print(f'{inst}: 失敗 ({msg[:200]}...)')
        # 失敗したエンドポイント・設定・モデルを片付ける
        for fn, kw in ((sm.delete_endpoint, {'EndpointName': endpoint_name}),
                       (sm.delete_endpoint_config, {'EndpointConfigName': endpoint_name}),
                       (sm.delete_model, {'ModelName': vllm_model.name})):
            try:
                fn(**kw)
            except Exception:
                pass
        if 'InsufficientInstanceCapacity' not in msg and 'ResourceLimitExceeded' not in msg:
            raise   # 在庫・クォータ以外の失敗は原因を見る (CloudWatch のコンテナログ)
assert predictor is not None, 'すべての候補で在庫かクォータが足りませんでした。時間をおいて再実行してください'


## 4. 呼び出し

`/invocations` に OpenAI Chat Completions 形式の JSON を送ります。学習時と同じく思考モードを無効にするため `chat_template_kwargs` を付けます。


In [ ]:
val_local = 'artifacts/val.jsonl'
os.makedirs('artifacts', exist_ok=True)
boto3.client('s3').download_file(bucket, 'automodel/cooking_basics/validation/val.jsonl', val_local)
prompts = [json.loads(l) for l in open(val_local, encoding='utf-8') if l.strip()][:5]

results = []
for r in prompts:
    resp = predictor.predict({
        'model': SERVED_NAME,
        'messages': [{'role': 'user', 'content': r['prompt']}],
        'max_tokens': 128,
        'temperature': 0,
        'chat_template_kwargs': {'enable_thinking': False},
    })
    text = resp['choices'][0]['message']['content']
    results.append({'prompt': r['prompt'], 'expected': r['output'], 'vllm': text, 'usage': resp.get('usage')})
    print('-' * 80); print('prompt  :', r['prompt']); print('expected:', r['output'][:120]); print('vllm    :', text)
json.dump(results, open(f'artifacts/vllm_{endpoint_name}.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=2)


## 5. ステップ1 の結果との比較（任意）

`02_verify_adapter_inference.ipynb` の `adapter_inference.json` があれば、同じプロンプトのアダプタ付き生成（HF）と vLLM の出力を並べます。
bf16 の演算順やカーネルの違いで完全一致はしませんが、文体と内容が同じ傾向であれば十分です。


In [ ]:
import glob
cands = sorted(glob.glob('artifacts/automodel-verify-adapter-*/adapter_inference.json'))
if cands:
    ref = json.load(open(cands[-1], encoding='utf-8'))
    hf = {g['prompt']: g['adapter'] for g in ref['generations']}
    for r in results:
        print('-' * 80); print('prompt:', r['prompt']); print('HF+PEFT:', hf.get(r['prompt'], '(なし)')[:160]); print('vLLM   :', r['vllm'][:160])
else:
    print('ステップ1 の結果ファイルが見つかりません (比較をスキップ)')


## 6. 後片付け（必ず実行）

In [ ]:
predictor.delete_endpoint(delete_endpoint_config=True)
vllm_model.delete_model()
print('deleted endpoint:', endpoint_name)
